# **Experiment Notebook**

## Weather Hazard Category — Experiment 1: Multinomial Logistic Baseline

In [ ]:
# Do not modify this code
!pip install -q utstd

from utstd.ipyrenders import *

In [ ]:
# Do not modify this code
import warnings
warnings.simplefilter(action='ignore')

In [ ]:
import pandas as pd

pd.__version__

---
## Student Information

In [ ]:
student_name = "Abhishek Chopda"
student_id = "25611094"

In [ ]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

In [ ]:
# Standard data science stack.
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Project toolkit, published to TestPyPI as `mlkit-aby`.
from mlkit_aby.helpers import (
    eda,
    features,
    importance,
    modeling,
    targets,
    timeseries,
    visualization,
    weather,
)

from weather_intelligence.config import (
    RAW_WEATHER_FILE,
    WHC_HORIZON,
    WHC_MODELS_DIR,
    WHC_TRAIN_START,
)

from sklearn.linear_model import LogisticRegression

pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

# The four hazard classes, in order. Passed explicitly to every metric so
# that a class absent from a split is still reported rather than silently
# dropped -- which matters enormously here, as section C.4 shows.
CLASS_LABELS = [0, 1, 2, 3]
CLASS_NAMES = [targets.WHC_LABELS[c] for c in CLASS_LABELS]

print("Forecast horizon:", WHC_HORIZON, "days")
print("Modelling window starts:", WHC_TRAIN_START)
print("Classes:", CLASS_NAMES)

---
## A. Project Description

In [ ]:
business_use_case_description = """
Open Meteo is adding predictive intelligence to its weather platform. This
experiment builds the second of two services: a seven-day-ahead forecast of
the Weather Hazard Category (WHC) for Sydney.

The WHC condenses rainfall, wind gusts, cloud cover, snowfall and
temperature into a hazard severity band: Low, Moderate, High or Extreme
Risk. Where the comfort service answers "is this a pleasant day to be
outdoors", this one answers a different and more consequential question:
"is this a day on which people, infrastructure or transport are at risk".

The seven-day horizon is the defining constraint. A one-day hazard warning
allows people to shelter. A seven-day warning allows organisations to act:
reschedule works, reposition crews, pre-position equipment, alter transport
timetables. The value of the service lies almost entirely in that lead time,
and so does its difficulty.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='business_use_case_description', value=business_use_case_description)

In [ ]:
business_objectives = """
The objective is to identify hazardous days far enough ahead to be acted
upon, accepting that precision will be sacrificed to achieve it.

The costs of the two error types are wildly asymmetric, far more so than for
the comfort service. A missed hazard -- predicting Low Risk for a day that
turns out to be High or Extreme -- means crews unprepared, works underway in
dangerous conditions, and transport running into a storm. A false alarm
means a day of unnecessary caution: postponed work, idle crews, and some
erosion of trust if it happens often.

The first is materially worse than the second. This has a direct
consequence for how the model should be built and judged: recall on the
hazardous classes matters far more than overall accuracy, and a model
should be tuned to over-warn rather than under-warn.

There is a hard limit on what is achievable, quantified in section C.4.
Extreme Risk days are vanishingly rare in Sydney, and no model trained on
so few examples can be expected to identify them reliably. Establishing
that limit honestly is itself a project objective, because a service that
silently never predicts its most severe class would be actively dangerous
if users assumed otherwise.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='business_objectives', value=business_objectives)

In [ ]:
stakeholders_expectations_explanations = """
Predictions are consumed through a REST endpoint accepting a date and
returning the hazard category exactly seven days later.

Direct users are developers integrating the endpoint into operational
scheduling and alerting systems. Indirectly affected are the organisations
acting on those alerts -- councils scheduling outdoor works, transport
operators, event organisers, utilities planning crew availability -- and
ultimately members of the public whose safety depends on those decisions.

Expectations differ sharply from the comfort service in one respect.
Comfort predictions are advisory and a wrong answer costs money. Hazard
predictions may inform safety decisions, and a wrong answer can cost more
than that. The service must therefore be explicit about its limitations
rather than presenting a bare category as authoritative: users need to know
which classes the model can actually detect, and that a seven-day hazard
forecast is a planning aid rather than a warning system. The
/model-metadata endpoint carries the per-class performance figures that make
this visible.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='stakeholders_expectations_explanations', value=stakeholders_expectations_explanations)

---
## C. Data Understanding

### C.1   Load Raw Data

In [ ]:
# The same hourly Open-Meteo record used for the comfort experiments. If it
# is absent, regenerate with:
#     uv run python -m weather_intelligence.dataset
hourly = pd.read_csv(RAW_WEATHER_FILE, parse_dates=["time"])

print(f"{len(hourly):,} hourly observations")
print(f"{hourly['time'].min()} to {hourly['time'].max()}")
hourly.head()

In [ ]:
eda.missing_summary(hourly)

### C.2 Define Target variable

In [ ]:
# The WHI formula is defined on daily values, so hourly observations are
# aggregated first. Note the aggregations relevant to this target: rainfall
# and snowfall are daily totals, and wind gusts are the daily MAXIMUM.
{k: v for k, v in weather.TARGET_DAILY_AGGREGATIONS.items()
 if k in ["precipitation", "wind_gusts_10m", "cloud_cover", "snowfall", "temperature_2m"]}

In [ ]:
daily = weather.aggregate_to_daily(hourly)
daily["whi"] = targets.weather_hazard_index(daily)
daily["whc"] = targets.weather_hazard_category(daily["whi"])

print(f"{len(daily):,} daily observations")
daily[["date", "precipitation", "wind_gusts_10m", "cloud_cover",
       "snowfall", "temperature_2m", "whi", "whc"]].head()

In [ ]:
target_definition_explanations = """
The target is the Weather Hazard Category, a four-level ordinal
classification derived by banding a continuous Weather Hazard Index. The WHI
is a weighted combination of five hazard scores, each capped at 1, combined
into a 0-100 index and then cut at 25, 50 and 75 to give Low, Moderate, High
and Extreme Risk. The prediction horizon is exactly seven days.

As with the comfort index, aggregation to daily values happens before the
formula is applied, because the component formulas are non-linear and the
constants are scaled to daily quantities. The aggregator per variable is
chosen to match what each hazard actually means. Rainfall and snowfall use
daily totals, since what floods a road is how much fell over the day. Wind
gusts use the daily MAXIMUM rather than the mean, because a hazard is
defined by the worst moment of the day: a single 95 km/h gust brings down a
tree regardless of how calm the other twenty-three hours were. Averaging
would dilute exactly the signal the target is meant to capture.

Two aspects of the specification deserve comment.

The model is required to predict the banded category rather than the
continuous index. This discards information -- a WHI of 49.9 and one of 25.1
are both Moderate Risk despite being very different days -- and it makes the
boundaries arbitrary in a way the underlying weather is not. It also turns a
naturally ordinal quantity into a nominal classification, so a standard
classifier treats confusing Low with Extreme as no worse than confusing Low
with Moderate, when operationally it is far worse.

The snowfall term is inapplicable here. Sydney records no snowfall in the
entire 26-year record, so that hazard is identically zero and 10% of the
index weight is a constant. The effective index is a weighted combination of
four terms, not five, and the maximum attainable WHI is 90 rather than 100.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='target_definition_explanations', value=target_definition_explanations)

### C.3 Create Target variable

In [ ]:
# Hazard components computed separately so each can be inspected.
whi_components = targets.weather_hazard_components(daily)
whi_components.describe().T.round(4)

In [ ]:
target_name = "whc"

daily["whc_label"] = targets.label_weather_hazard_category(daily["whc"])
daily[["date", "whi", "whc", "whc_label"]].head()

### C.4 Explore Target variable

In [ ]:
# The single most important table in this notebook.
distribution = daily["whc"].value_counts().reindex(CLASS_LABELS, fill_value=0)
summary = pd.DataFrame({
    "class": CLASS_LABELS,
    "label": CLASS_NAMES,
    "days": distribution.to_numpy(),
    "share": (distribution / len(daily)).to_numpy(),
})
summary["one_day_every"] = (len(daily) / distribution.replace(0, np.nan)).to_numpy()
summary.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(daily["whi"], bins=60, edgecolor="white")
for boundary in [25, 50, 75]:
    axes[0].axvline(boundary, linestyle="--", color="black")
axes[0].set_title("Distribution of the Weather Hazard Index")
axes[0].set_xlabel("WHI")
axes[0].set_ylabel("Number of days")

axes[1].bar(CLASS_NAMES, distribution.to_numpy())
axes[1].set_yscale("log")
axes[1].set_title("Weather Hazard Category (log scale)")
axes[1].set_ylabel("Number of days")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# Every Extreme Risk day in 26 years, with the conditions that produced it.
extreme_days = daily[daily["whc"] == 3]
extreme_days[["date", "precipitation", "wind_gusts_10m", "cloud_cover",
              "temperature_2m", "whi"]].round(1).to_string(index=False)

In [ ]:
# Does the snowfall term contribute anything at all?
print("Days with any snowfall:", int((daily["snowfall"] > 0).sum()))
print("Maximum attainable WHI given zero snowfall:", 100 * (0.30 + 0.30 + 0.20 + 0.10))
print("Highest WHI observed:", round(daily["whi"].max(), 1))
print()
# How much does each hazard term actually vary?
contribution = pd.DataFrame({
    "std": whi_components.std(),
    "weight": pd.Series(targets.WHI_WEIGHTS),
})
contribution["weighted_std"] = contribution["std"] * contribution["weight"]
contribution.sort_values("weighted_std", ascending=False).round(4)

In [ ]:
# Hazard is seasonal too, but differently from comfort.
by_month = (
    daily.assign(month=daily["date"].dt.month)
    .groupby("month")["whc"]
    .apply(lambda s: (s >= 2).mean())
)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(by_month.index, by_month.to_numpy())
ax.set_xticks(range(1, 13))
ax.set_xlabel("Month")
ax.set_ylabel("Share of days at High or Extreme Risk")
ax.set_title("Hazardous days by calendar month")
plt.tight_layout()
plt.show()

by_month.round(4)

In [ ]:
target_distribution_explanations = """
The class distribution is the defining problem of this experiment, and it is
severe.

Across 9,497 days, Low Risk accounts for 5,242 days and Moderate Risk for
3,970 -- together 97% of the record. High Risk occurs on 274 days, roughly
2.9%. Extreme Risk occurs on 11 days in 26 years, which is one day every
2.4 years and 0.12% of the data.

Three consequences follow, and they shape everything downstream.

First, accuracy is a worthless metric here. A model that predicts Low Risk
for every single day would be 55% accurate while providing no value
whatsoever, and one that never predicts anything above Moderate would score
97%. Macro F1, balanced accuracy and per-class recall are used instead,
because they weight every class equally and expose exactly this failure.

Second, extending the historical record barely helps. Restricting to
2022-2025 gives 3 Extreme days; using the full 26 years gives 11. The
rarity is a property of Sydney's climate, not of the sample size, so it
cannot be solved by collecting more of the same data. This is why the
modelling window for this target uses the entire record while the comfort
model uses only 2015 onwards -- here every rare example is worth more than
the cost of the distribution drift it brings.

Third, and most importantly, no model trained on 11 examples split across
train, validation and test can be expected to identify Extreme Risk
reliably. This should be stated as a finding rather than discovered as a
disappointment. The realistic objective is to detect High Risk days with
useful recall and to treat Extreme Risk as a class the model can represent
but not reliably predict.

The eleven Extreme days share a clear and consistent signature: heavy
rainfall above 60 mm, severe gusts above 75 km/h, and near-total cloud
cover, occurring together. They are identifiable storm events -- the
February 2020 east coast low with 138 mm of rain and 102 km/h gusts, the
March 2022 floods. They are not noise or data errors, which is encouraging:
the pattern is learnable in principle, there is simply almost nothing to
learn it from.

The snowfall term contributes nothing. No day in the record records any
snowfall, so that hazard is constant at zero, 10% of the index weight
carries no information, and the maximum attainable WHI is 90 rather than
100. The highest observed value is 83.1. In practice the index is driven by
rainfall and wind gusts, which carry 0.30 each and vary substantially, with
cloud cover contributing a moderate amount and temperature very little.

Hazard is seasonal, but the pattern differs from comfort. High and Extreme
Risk days cluster in the cooler, wetter months rather than distributing
evenly, which suggests the calendar features will carry genuine signal for
this target as well.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='target_distribution_explanations', value=target_distribution_explanations)

### C.5 Explore Feature of Interest `wind_gusts_10m`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(daily["wind_gusts_10m"], bins=60, edgecolor="white")
axes[0].axvline(100, linestyle="--", color="black", label="WHI gust cap (100 km/h)")
axes[0].set_title("Distribution of daily maximum wind gusts")
axes[0].set_xlabel("Gust (km/h)")
axes[0].legend()

for cls, name in zip(CLASS_LABELS, CLASS_NAMES):
    subset = daily.loc[daily["whc"] == cls, "wind_gusts_10m"]
    if len(subset):
        axes[1].scatter([cls] * len(subset), subset, alpha=0.2, edgecolor="none")
axes[1].set_xticks(CLASS_LABELS)
axes[1].set_xticklabels(CLASS_NAMES, rotation=20)
axes[1].set_ylabel("Gust (km/h)")
axes[1].set_title("Daily maximum gust by hazard class")

plt.tight_layout()
plt.show()

daily.groupby("whc")["wind_gusts_10m"].describe()[["count", "mean", "min", "max"]].round(1)

In [ ]:
feature_1_insights = """
Daily maximum wind gusts carry a weight of 0.30 in the index, equal highest
with rainfall, and vary widely, making them one of the two dominant drivers.

The distribution is right skewed but far less extremely so than rainfall.
Most days peak somewhere between 25 and 55 km/h, with a tail extending past
100. The 100 km/h cap in the formula is reached only a handful of times in
26 years, so unlike the rain term the gust hazard is almost never saturated
and varies smoothly across nearly its whole range.

The separation by hazard class is clear and monotonic: mean gust speed rises
consistently from Low through to Extreme Risk. That is partly circular,
since gusts help define the class, but it confirms the term is doing real
work rather than being swamped by the others.

The more useful observation for forecasting is that gusts are less
persistent day to day than temperature. Knowing today's peak gust says
relatively little about the peak gust seven days from now, which is the
horizon this model must span. The predictive value is more likely to lie in
rolling summaries -- whether the recent period has been unsettled -- than in
any single lagged value.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### C.6 Explore Feature of Interest `precipitation`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(daily["precipitation"], bins=60, edgecolor="white")
axes[0].set_yscale("log")
axes[0].axvline(30, linestyle="--", color="black", label="WHI rain cap (30 mm)")
axes[0].set_title("Distribution of daily rainfall (log count)")
axes[0].set_xlabel("Precipitation (mm)")
axes[0].legend()

axes[1].boxplot(
    [daily.loc[daily["whc"] == c, "precipitation"] for c in CLASS_LABELS],
    tick_labels=CLASS_NAMES,
)
axes[1].set_ylabel("Precipitation (mm)")
axes[1].set_title("Daily rainfall by hazard class")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

print("Days at or above the 30 mm cap: {} ({:.2%})".format(
    int((daily["precipitation"] >= 30).sum()), (daily["precipitation"] >= 30).mean()))

In [ ]:
feature_2_insights = """
Rainfall shares the highest weight in the index at 0.30 and is the most
awkwardly distributed variable in the dataset. Just over half of all days
record no rain at all, and the distribution decays steeply from there.

Its relationship to hazard is a clipped ramp: rainfall above 30 mm saturates
the rain hazard at 1, and beyond that point further rain changes the index
not at all. Roughly 1.5% of days reach that cap. The class boxplots show the
expected progression, but with wide overlap between Low and Moderate Risk --
the majority of days in both classes are dry, and it is the other terms that
separate them.

For a seven-day forecast this variable is doubly difficult. Its distribution
is dominated by zeros, which suits neither a linear model nor a distance
based one, and rainfall is among the least persistent weather variables:
whether it rains on a given day says very little about whether it will rain
a week later. The realistic expectation is that raw lagged rainfall carries
almost no seven-day signal, and that any usable information comes from
longer rolling aggregates capturing whether the period is broadly wet -- a
wet synoptic pattern persists even when individual rain days do not.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### C.n Explore Feature of Interest

> Remaining variables are examined through the correlation analysis in section D.

In [ ]:
hazard_variables = ["precipitation", "wind_gusts_10m", "cloud_cover",
                    "temperature_2m", "relative_humidity_2m", "wind_speed_10m"]
eda.numerical_summary(daily[hazard_variables]).round(2)

---
## D. Feature Selection

### D.1 Approach "Correlation with the future target"

In [ ]:
# Which of today's variables carry information about the hazard SEVEN days
# from now? Correlation against the future WHI is used rather than the
# banded category, because banding discards the gradations that make a
# correlation meaningful.
future_whi = daily["whi"].shift(-WHC_HORIZON)

seven_day = daily[hazard_variables + ["whi"]].corrwith(future_whi)
same_day = daily[hazard_variables + ["whi"]].corrwith(daily["whi"])

pd.DataFrame({"same_day": same_day, f"{WHC_HORIZON}_days_ahead": seven_day}) \
    .sort_values("same_day", key=abs, ascending=False).round(3)

In [ ]:
feature_selection_1_insights = """
This comparison is the most sobering result in the notebook, and it should
be read before any modelling decision is made.

Same-day correlations are strong, as they must be -- these variables define
the index. Seven days ahead, they very nearly vanish. Every correlation
collapses towards zero, including today's WHI itself, which is the strongest
same-day predictor by construction.

The reason is meteorological rather than methodological. Individual weather
systems pass through Sydney over two to four days. At a seven-day horizon
the specific system responsible for today's conditions has long since
cleared, and the one that will determine conditions in a week has not yet
formed. Persistence, which carried the comfort model at one day ahead, is
essentially exhausted at seven.

This sets a realistic ceiling on what any model can achieve here, and it
distinguishes this problem sharply from the comfort forecast. Whatever
signal exists must come from slower-moving structure -- seasonal position,
and rolling summaries describing whether the current period is broadly
unsettled -- rather than from the state of any individual recent day. The
feature engineering in section F is built around that expectation, and the
comparison against a seasonal baseline in section J.4 is essential rather
than decorative: it is entirely possible that a model adds little beyond
knowing the time of year.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_selection_1_insights', value=feature_selection_1_insights)

### D.2 Approach "Variance and redundancy screening"

In [ ]:
# A feature that never varies cannot discriminate between classes.
zero_variance = daily[["snowfall"] + hazard_variables].std()
print("Standard deviation of each candidate:")
print(zero_variance.round(4).to_string())
print()
eda.find_highly_correlated_features(daily[hazard_variables], threshold=0.70)

In [ ]:
feature_selection_2_insights = """
This screen removes candidates that cannot contribute, either because they
do not vary or because another feature already carries their information.

Snowfall has a standard deviation of exactly zero. It is constant across
every one of the 9,497 days, so it cannot discriminate between any two days
and is excluded. This is worth stating explicitly rather than quietly
dropping, because it means 10% of the specified index weight is inert for
this location: the WHI as applied to Sydney is effectively a four-term index.

Wind speed and wind gusts are strongly correlated, measuring the same
phenomenon at different statistics. Unlike the comfort model, here the
formula uses gusts, so gusts are retained as the primary variable. Mean wind
speed is nonetheless kept as a candidate, because the two carry slightly
different information at a seven-day horizon: mean wind speed is a
smoother, more persistent quantity than peak gust and may survive the
horizon better.

Relative humidity is retained despite not appearing in the WHI formula. It
does not define the target, but it is informative about whether the
atmosphere is in a wet regime, and at seven days that regime-level
information is more likely to survive than any specific variable. This is a
deliberate departure from the comfort model, where features were restricted
to the variables composing the index.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_selection_2_insights', value=feature_selection_2_insights)

## D.z Final Selection of Features

In [ ]:
# Variables from which lag and rolling features are derived. Snowfall is
# excluded as constant; humidity and mean wind speed are included as regime
# indicators despite not appearing in the WHI formula.
features_list = [
    "precipitation",
    "wind_gusts_10m",
    "cloud_cover",
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
]

base_features = features_list + ["whi"]
base_features

In [ ]:
feature_selection_explanations = """
Six weather variables are selected, plus the continuous WHI as a
persistence-style signal.

The four variables the formula actually uses and can vary -- rainfall, wind
gusts, cloud cover and temperature -- are included because they define the
target. Snowfall is excluded as identically zero.

Two variables outside the formula are added. Mean wind speed is smoother and
more persistent than peak gust, and relative humidity indicates whether the
atmosphere is in a broadly wet regime. Both are included specifically
because section D.1 showed that day-specific signal does not survive seven
days: what might survive is regime-level information, and these two describe
the regime better than the sharp, spiky variables the formula uses.

The continuous WHI is included rather than the banded category. It is
computed entirely from the same day's weather so it is known at prediction
time, and it preserves gradations that banding would discard -- a day at 49
and a day at 26 are both Moderate Risk but carry very different information
about the days around them.

As with the comfort experiment, these are base variables rather than final
model inputs. Section F derives the lag, rolling and calendar features the
model actually receives.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## E. Data Preparation

### E.1 Data Transformation: Completeness and continuity

In [ ]:
expected_dates = pd.date_range(daily["date"].min(), daily["date"].max(), freq="D")

print("Missing values in daily frame:", int(daily.isna().sum().sum()))
print("Missing calendar days:", len(expected_dates.difference(daily["date"])))
print("Duplicate dates:", int(daily["date"].duplicated().sum()))

In [ ]:
data_cleaning_1_explanations = """
The daily record is complete: no missing values, no missing calendar days
and no duplicate dates across the full 26 years.

Continuity is the critical property here, more so than for the comfort
model, because this target uses a seven-day horizon. Every lag, rolling
window and the target shift itself are positional operations: they move
rows, not dates. A single missing day would mean the "seven days ahead"
target silently became eight days ahead from that point onward, and a
rolling thirty-day window would span thirty-one calendar days. Neither would
raise an error or produce a missing value; both would quietly corrupt the
data in a way that is extremely difficult to detect downstream.

Confirming continuity is what licenses the positional approach. Had gaps
been present, the frame would first have had to be reindexed onto a complete
date range so that the gaps became explicit missing values.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### E.2 Data Transformation: Target integrity

In [ ]:
# The target is produced by code, so it is verified against its definition
# rather than assumed correct.
print("WHI within 0-100:", bool(daily["whi"].between(0, 100).all()))
print("WHC values present:", sorted(daily["whc"].unique().tolist()))

# Every banding boundary must behave as specified: left-closed, right-open.
boundary_check = pd.DataFrame({
    "whi": [0.0, 24.999, 25.0, 49.999, 50.0, 74.999, 75.0, 90.0],
})
boundary_check["whc"] = targets.weather_hazard_category(boundary_check["whi"])
boundary_check

In [ ]:
data_cleaning_2_explanations = """
The target is constructed by code rather than measured, so it is verified
against its specification instead of being assumed correct.

Three checks are applied. The WHI falls within its defined 0-100 range for
every day. All four categories appear at least once, so the problem is
genuinely four-class even though one class is vanishingly rare. And the
banding boundaries behave as specified: a WHI of exactly 25 is classified
Moderate Risk rather than Low, confirming the bands are left-closed and
right-open as the specification requires.

That last check is not pedantic. An off-by-one at a boundary would
misclassify every day landing exactly on 25, 50 or 75, and with continuous
weather data such days are rare enough that the error could persist
unnoticed while subtly corrupting the rare-class counts that matter most
here. The same property is asserted in the package's unit tests; this
repeats the check against the real data.

No transformation is applied. This is a validation step and the correct
outcome is that nothing needed fixing.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### E.3 Data Transformation: Class imbalance strategy

In [ ]:
# Rather than resampling the data, imbalance is handled at the model through
# class weights. The weights below are what "balanced" computes.
counts = daily["whc"].value_counts().reindex(CLASS_LABELS, fill_value=0)
balanced_weights = len(daily) / (len(CLASS_LABELS) * counts.replace(0, np.nan))

pd.DataFrame({
    "label": CLASS_NAMES,
    "days": counts.to_numpy(),
    "balanced_weight": balanced_weights.round(1).to_numpy(),
})

In [ ]:
data_cleaning_3_explanations = """
Class imbalance is handled at the model, through class weights, rather than
in the data through resampling.

The weights that `class_weight="balanced"` computes are shown above: an
Extreme Risk day is weighted roughly two hundred times more heavily than a
Low Risk day. Each of the eleven Extreme examples therefore contributes to
the loss as though it were hundreds of days, which is what prevents the
model from simply ignoring the class.

Resampling was considered and rejected. Oversampling the rare classes, by
duplication or by a synthetic method such as SMOTE, would generate new rows
that do not correspond to real days. In a time series this is doubly
problematic: synthetic days have no place in the chronological ordering, and
interpolating between two storm events two years apart produces a "day" that
never occurred and could not occur. Undersampling the majority classes would
discard the overwhelming majority of the record to accommodate eleven days.
Class weighting achieves the same rebalancing of influence without
fabricating or destroying data.

The limitation must be stated honestly: class weighting changes how heavily
the eleven Extreme days count, but it does not create additional examples.
Eleven days weighted heavily is still eleven days. The technique makes the
class visible to the optimiser rather than making it learnable, and section
J.4 measures how far that gets us. A predictable side effect is a large
increase in false alarms, since the model is now heavily penalised for
missing rare classes and barely penalised for over-predicting them -- which,
given the cost asymmetry set out in section A, is the direction we would
choose anyway.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

---
## F. Feature Engineering

### F.1 New Feature "Recipe-driven feature construction"

In [ ]:
# The experiment-specific decision is which features to build; the package
# assembles them. The recipe is saved inside the model artefact so the
# deployed API builds features identically.
#
# Windows are longer than in the comfort experiment. Section D.1 showed that
# day-specific signal does not survive seven days, so short windows are
# unlikely to help and longer ones describing the prevailing regime are
# more promising.
WHC_RECIPE = {
    "base_variables": base_features,
    "lags": [1, 2, 3, 7],
    "windows": [7, 14, 30, 60],
    "statistics": ["mean", "std"],
    "derived_indices": ["whi"],
    "within_day": True,
    "within_day_lags": [1],
}

daily_features = features.build_daily_features(hourly, WHC_RECIPE)

print(f"{daily_features.shape[1] - 1} features for {len(daily_features):,} days")
print(f"History required per prediction: {features.required_history_days(WHC_RECIPE)} days")
daily_features[["date", "whi", "whi_lag1", "whi_roll7_mean", "whi_roll60_mean"]].tail(5)

In [ ]:
feature_engineering_1_explanations = """
The feature set is declared as a recipe and assembled by the package, which
serves two purposes: the deployed API builds its features from the same
description, and the choices that define this experiment are visible in one
place rather than spread across cells.

Three choices distinguish this recipe from the comfort experiment's.

The rolling windows are longer, running to sixty days rather than thirty.
Section D.1 established that individual weather variables retain almost no
correlation with conditions seven days ahead, so features describing a
specific recent day have little to offer. What may survive the horizon is
the prevailing regime: whether the last fortnight or two months have been
unsettled. Longer windows describe that; short ones do not.

The lags are retained despite the same argument, deliberately, as a control.
If the short lags turn out to carry no weight in the fitted model, that is
itself a useful confirmation of the D.1 finding rather than a wasted
feature.

The derived index is the WHI rather than the CCI, and it is included at
every lag and window. Its rolling means act as a hazard climatology, giving
the model a sense of how hazardous the current period has been in aggregate
even when no individual day predicts the target.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### F.2 New Feature "Rolling regime summaries"

In [ ]:
rolling_columns = [c for c in daily_features.columns if "_roll" in c]
lag_columns = [c for c in daily_features.columns if "_lag" in c]
print(f"{len(rolling_columns)} rolling features, {len(lag_columns)} lag features")

# Do the longer windows actually carry more seven-day signal than the raw
# values? This is the D.1 question asked of the engineered features.
future_whi = daily_features["whi"].shift(-WHC_HORIZON)
candidates = ["whi", "whi_lag7", "whi_roll7_mean", "whi_roll14_mean",
              "whi_roll30_mean", "whi_roll60_mean",
              "precipitation_roll30_mean", "wind_gusts_10m_roll30_mean"]
daily_features[candidates].corrwith(future_whi).sort_values(key=abs, ascending=False).round(3)

In [ ]:
feature_engineering_2_explanations = """
This is the direct test of the hypothesis that motivated the longer windows,
and it is the most informative single output in the feature engineering
section.

The comparison ranks raw and rolling features by their correlation with the
WHI seven days ahead. If the regime hypothesis holds, the rolling means
should correlate more strongly with the future than today's raw value does,
despite summarising older information. Today's WHI describes one specific
day whose weather system will have cleared within a week; a thirty-day mean
describes a prevailing state that changes far more slowly and is therefore
more likely to persist across the horizon.

Whatever the result, it directly informs the next experiment. If the longer
windows dominate, feature engineering should push further in that direction
with seasonal aggregates and longer-range indicators. If they do not, then
very little exists in the recent record at this horizon, the calendar
features are doing most of the work, and effort is better spent on external
predictors -- synoptic indices, pressure patterns, oceanic oscillation
indices -- than on further summaries of the same local observations.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### F.3 New Feature "Cyclical calendar encoding"

In [ ]:
# Section C.4 showed hazardous days cluster seasonally, and the calendar is
# the one input known with certainty for any future date -- which matters
# more at seven days than at one.
year_boundary = daily_features[
    daily_features["date"].isin(
        pd.to_datetime(["2024-12-30", "2024-12-31", "2025-01-01", "2025-01-02"])
    )
]
year_boundary[["date", "day_of_year_sin", "day_of_year_cos"]].round(4)

In [ ]:
feature_engineering_n_explanations = """
Day of year and month are encoded as sine and cosine pairs, placing each
date on a circle so that 31 December and 1 January sit adjacent rather than
364 units apart, as the output above shows.

These features matter more for this target than for the comfort forecast,
for a reason specific to the horizon. At seven days ahead the recent weather
has largely lost its predictive value, while the calendar has lost none of
it: the date of a day a week away is known exactly, and section C.4 showed
that High and Extreme Risk days cluster in the cooler, wetter months rather
than occurring uniformly through the year.

The likely outcome is that this model leans on the calendar considerably
more heavily than the comfort model did, and the comparison against a purely
seasonal baseline in section J.4 is therefore essential. If the model cannot
beat a baseline that knows only the time of year, then the recent weather
features are contributing nothing at this horizon, and that is a finding
worth reporting rather than concealing.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_n_explanations', value=feature_engineering_n_explanations)

---
## G. Data Preparation for Modeling

### G.1 Split Datasets

In [ ]:
# Features were built on the full record so rolling windows are complete
# from the first modelled day. This target uses the entire window.
model_frame = daily_features[
    daily_features["date"] >= WHC_TRAIN_START
].reset_index(drop=True)

EXCLUDED_COLUMNS = {"date", "target_date", "whc", "whc_h7", "whc_label"}
feature_columns = [c for c in model_frame.columns if c not in EXCLUDED_COLUMNS]

print(f"Window: {model_frame['date'].min().date()} to {model_frame['date'].max().date()}")
print(f"{len(model_frame):,} days, {len(feature_columns)} candidate features")
print("Today's WHI retained as an input:", "whi" in feature_columns)

In [ ]:
# Split boundaries chosen so that every split contains at least some
# Extreme Risk days. With only eleven in the record this is a real
# constraint on where the boundaries can fall, not a free choice.
VALIDATION_START = "2018-01-01"
TEST_START = "2022-01-01"

supervised = timeseries.make_supervised_frame(
    model_frame, "whc", horizon=WHC_HORIZON
)
supervised = supervised.dropna(subset=feature_columns).reset_index(drop=True)

# The shift introduces missing values and so returns a float column; the
# target is categorical and must be restored to an integer type.
supervised["whc_h7"] = supervised["whc_h7"].astype(int)

train, validation, test = timeseries.chronological_split(
    supervised,
    validation_start=VALIDATION_START,
    test_start=TEST_START,
    horizon=WHC_HORIZON,
)

X_train, y_train = train[feature_columns], train["whc_h7"]
X_val, y_val = validation[feature_columns], validation["whc_h7"]
X_test, y_test = test[feature_columns], test["whc_h7"]

print(f"train      {len(train):>5}   {train['date'].min().date()} -> {train['date'].max().date()}")
print(f"validation {len(validation):>5}   {validation['date'].min().date()} -> {validation['date'].max().date()}")
print(f"test       {len(test):>5}   {test['date'].min().date()} -> {test['date'].max().date()}")

In [ ]:
# Class counts per split. With eleven Extreme days in total, this table
# determines what can and cannot be measured.
split_counts = pd.DataFrame({
    name: part["whc_h7"].value_counts().reindex(CLASS_LABELS, fill_value=0)
    for name, part in [("train", train), ("validation", validation), ("test", test)]
})
split_counts.index = CLASS_NAMES
split_counts

In [ ]:
# The embargo must hold: no training target may fall inside validation.
assert train["target_date"].max() < pd.Timestamp(VALIDATION_START)
assert validation["target_date"].max() < pd.Timestamp(TEST_START)
assert train["date"].max() < validation["date"].min()
assert validation["date"].max() < test["date"].min()
print("Embargo verified: no target crosses a split boundary.")

In [ ]:
data_splitting_explanations = """
The split is chronological, with roughly eighteen years for training and
four years each for validation and test.

The reasoning for splitting by time rather than at random is the same as for
the comfort model, and applies with more force here. Weather is
autocorrelated, so a random split would place near-identical adjacent days
on both sides of the boundary and inflate the score. With rare classes the
distortion would be worse still: the eleven Extreme days occur as
consecutive pairs in two cases, so a random split could put 8 June 2007 in
training and 9 June 2007 in test, and the model would appear to detect an
Extreme event it had effectively already been shown.

An embargo of seven days is applied at each boundary, removing rows whose
targets fall on the far side. At this horizon the embargo matters more than
it did for the comfort model, since seven rows of leakage is seven days of
future information rather than one.

The boundary placement is constrained rather than free. With only eleven
Extreme days in 26 years, most boundary choices would leave one or more
splits with none at all, making the class unmeasurable in that split. The
2018 and 2022 boundaries were chosen because they distribute those days
across all three splits while keeping the periods a sensible size. This is
worth acknowledging as a mild methodological compromise: the boundaries were
informed by the target distribution rather than chosen blind. The
alternative -- a test set containing no examples of the class the service is
most expected to detect -- would have been considerably worse.

Even so, the counts per split are very small, and any per-class metric for
Extreme Risk is computed from a handful of days. Such figures are reported
for completeness but carry almost no statistical weight, and section J.4
treats them accordingly.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_splitting_explanations', value=data_splitting_explanations)

### G.2 Data Transformation "Standardisation within the model pipeline"

In [ ]:
# As with the comfort experiment, scaling is the first stage of a pipeline
# rather than a separate step, so the scaler and model are fitted, saved
# and applied together.
def make_model(C=1.0, class_weight="balanced"):
    """Build an unfitted scaler + multinomial logistic pipeline."""
    return modeling.make_scaled_pipeline(
        LogisticRegression(
            C=C,
            class_weight=class_weight,
            max_iter=5000,
            random_state=RANDOM_STATE,
        )
    )


demonstration = make_model().fit(X_train, y_train)
scaled = demonstration.named_steps["scaler"].transform(X_train)
pd.DataFrame(scaled, columns=feature_columns)[
    ["precipitation", "wind_gusts_10m"]
].describe().loc[["mean", "std"]].round(3)

In [ ]:
data_transformation_1_explanations = """
Features are standardised as the first stage of the model pipeline.

Logistic regression with an L2 penalty requires this for the same reason
ridge regression does: the penalty acts on coefficient magnitude, which
depends on each feature's units, so without a common scale the amount of
regularisation applied to rainfall in millimetres and to humidity in percent
would differ arbitrarily. Standardising also makes the fitted coefficients
comparable across features.

There is an additional reason specific to this model. Logistic regression is
fitted by iterative optimisation rather than a closed-form solution, and
convergence is markedly slower when features span very different scales.
With 130 features and severe class weighting the optimiser is already
working hard, and unscaled inputs would make convergence unreliable.

The scaler is fitted on the training split alone and reused unchanged
thereafter. Fitting it on all data before splitting would encode information
about the test period into the training statistics, which in a time series
means encoding the future. Placing it inside the pipeline enforces this
automatically and guarantees that the deployed artefact applies exactly the
transformation the model was trained with.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

---
## H. Save Datasets

In [ ]:
data_final_path = "../../data/processed"
Path(data_final_path).mkdir(parents=True, exist_ok=True)
print("Saving WHC modelling datasets to", data_final_path)

In [ ]:
# Do not modify this code
# Save training set
try:
  X_train.to_csv(f'{data_final_path}/X_train.csv', index=False)
  y_train.to_csv(f'{data_final_path}/y_train.csv', index=False)

  X_val.to_csv(f'{data_final_path}/X_val.csv', index=False)
  y_val.to_csv(f'{data_final_path}/y_val.csv', index=False)

  X_test.to_csv(f'{data_final_path}/X_test.csv', index=False)
  y_test.to_csv(f'{data_final_path}/y_test.csv', index=False)
except Exception as e:
  print(e)

In [ ]:
# Saved again under target-specific names, so that the comfort and hazard
# datasets do not overwrite one another in the shared processed folder.
for name, frame in [("X_train", X_train), ("y_train", y_train),
                    ("X_val", X_val), ("y_val", y_val),
                    ("X_test", X_test), ("y_test", y_test)]:
    frame.to_csv(f"{data_final_path}/whc_{name}.csv", index=False)
print("Saved WHC-prefixed copies.")

---
## I. Selection of Performance Metrics

In [ ]:
help(modeling.evaluate_classification)

In [ ]:
# Why accuracy is excluded: the score a model achieves by never predicting
# anything hazardous at all.
naive_always_low = pd.Series([0] * len(y_test))
naive_metrics = modeling.evaluate_classification(y_test, naive_always_low, labels=CLASS_LABELS)

print(f"A model that always predicts 'Low Risk':")
print(f"  accuracy          {naive_metrics['accuracy']:.3f}")
print(f"  balanced accuracy {naive_metrics['balanced_accuracy']:.3f}")
print(f"  macro F1          {naive_metrics['macro_f1']:.3f}")

In [ ]:
performance_metrics_explanations = """
Macro F1 is the headline metric, supported by balanced accuracy and by
per-class recall. Accuracy is reported only to demonstrate why it must not
be used.

The output above makes the case directly. A model that predicts Low Risk for
every day in the test set -- a model containing no information whatsoever --
achieves a respectable accuracy, because Low Risk days are the plurality.
Its balanced accuracy is 0.25 and its macro F1 far lower, because both
metrics average performance across classes rather than across days, so
three classes scoring zero cannot be hidden behind one scoring well.

Per-class recall is reported for every class individually and is the figure
that matters operationally. Recall on High and Extreme Risk answers the
question the service exists to answer: of the genuinely hazardous days, how
many did we flag? Section A established that a missed hazard is far more
costly than a false alarm, which makes recall on the hazardous classes more
important than precision on them. A model that flags fifty days to catch
twenty real hazards may be entirely acceptable; one that flags five and
catches three is not.

Macro F1 is preferred to balanced accuracy as the single headline because it
accounts for precision as well as recall. Balanced accuracy alone can be
driven upwards by indiscriminate over-warning, and with heavy class
weighting that is a real risk rather than a theoretical one.

One caveat applies throughout. Per-class metrics for Extreme Risk are
computed from a handful of test days, so they are extremely unstable: a
single day changing outcome moves the recall by a large fraction. They are
reported for completeness and should not be read as reliable estimates.

A metric that recognised the ordering of the classes -- treating a Low for
Extreme confusion as worse than Low for Moderate -- would fit the problem
better than any of these. The specification requires a standard multiclass
classification, so this is noted as a limitation rather than acted upon, and
the confusion matrix in section J.4 is used to inspect the severity of
misclassifications directly.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='performance_metrics_explanations', value=performance_metrics_explanations)

## J. Train Machine Learning Model

### J.1 Import Algorithm

In [ ]:
help(modeling.make_scaled_pipeline)

In [ ]:
algorithm_selection_explanations = """
This experiment uses multinomial logistic regression with balanced class
weights, chosen as a baseline rather than as the expected final model. It is
the classification counterpart of the ridge regression used for the comfort
baseline, which keeps the two experiment tracks comparable.

Three reasons make it the right starting point.

It handles the class imbalance directly. The `class_weight="balanced"`
parameter rescales each class's contribution to the loss in inverse
proportion to its frequency, so the eleven Extreme Risk days are weighted
roughly two hundred times more heavily than a typical Low Risk day. Without
this the optimiser would find that ignoring the rare classes entirely is the
loss-minimising solution.

It produces calibrated class probabilities rather than bare labels. That
matters operationally: a hazard service that can say "35% chance of High
Risk" supports better decisions than one that can only say "Moderate", and
it leaves open the option of tuning the decision threshold towards
over-warning, which section A argued is the correct direction given the cost
asymmetry.

It is interpretable and fast, so it establishes a defensible floor cheaply.

Its limitations are known and are the reason this is experiment one. It
fits linear decision boundaries in feature space, and the conditions
producing an Extreme Risk day are a conjunction -- heavy rain AND severe
gusts AND total cloud, simultaneously -- which is an interaction a linear
model cannot represent without being given the interaction terms explicitly.
A tree ensemble represents conjunctions natively and is the obvious next
experiment. Establishing how far a linear model gets is what makes that
comparison meaningful.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='algorithm_selection_explanations', value=algorithm_selection_explanations)

### J.2 Set Hyperparameters

In [ ]:
# C is the inverse of regularisation strength: smaller values regularise
# harder. Selected on the validation set by macro F1, not accuracy.
C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0]

search = []
for C in C_GRID:
    for weighting in ["balanced", None]:
        candidate = make_model(C=C, class_weight=weighting).fit(X_train, y_train)
        metrics = modeling.evaluate_classification(
            y_val, candidate.predict(X_val), labels=CLASS_LABELS
        )
        search.append({
            "C": C,
            "class_weight": weighting or "none",
            "macro_f1": metrics["macro_f1"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            "accuracy": metrics["accuracy"],
            "recall_high": metrics["recall_class_2"],
            "recall_extreme": metrics["recall_class_3"],
        })

search = pd.DataFrame(search)
search.round(3)

In [ ]:
# Best configuration by validation macro F1.
best = search.loc[search["macro_f1"].idxmax()]
BEST_C = float(best["C"])
BEST_WEIGHT = None if best["class_weight"] == "none" else "balanced"

print(f"Selected: C={BEST_C}, class_weight={BEST_WEIGHT}")
print(f"Validation macro F1 {best['macro_f1']:.3f}, accuracy {best['accuracy']:.3f}")

In [ ]:
hyperparameters_selection_explanations = """
Two hyperparameters are tuned: the regularisation strength C, and whether
class weighting is applied at all.

C is the inverse of the penalty strength, so smaller values regularise more
heavily. It is searched over a logarithmic grid because the useful range
spans orders of magnitude. With 130 features and only a few hundred examples
in the minority classes, strong regularisation is expected to help: the
model has ample capacity to fit noise around the rare classes.

Class weighting is included in the search as a deliberate comparison rather
than assumed. Section E.3 argued for it on principle, and the search table
shows what it actually costs and buys: the unweighted model should post
higher raw accuracy while scoring far worse on macro F1 and on recall for
the hazardous classes. Demonstrating that trade-off with measurements is
more convincing than asserting it, and it directly supports the choice of
metric made in section I.

Selection is made on validation macro F1 rather than accuracy, for the
reasons set out in section I. Selecting on accuracy would reliably choose
the unweighted model and defeat the purpose of the exercise.

Tuning uses the dedicated validation period rather than k-fold
cross-validation, because shuffling folds would break the temporal ordering
and reintroduce the leakage the chronological split prevents. This costs
statistical power that would be valuable given how few rare examples exist,
but the alternative produces optimistic and untrustworthy estimates.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='hyperparameters_selection_explanations', value=hyperparameters_selection_explanations)

### J.3 Fit Model

In [ ]:
model = make_model(C=BEST_C, class_weight=BEST_WEIGHT).fit(X_train, y_train)

print(f"Fitted multinomial logistic regression (C={BEST_C}, "
      f"class_weight={BEST_WEIGHT}) on {len(X_train):,} days "
      f"x {len(feature_columns)} features")
print("Classes learned:", model.named_steps["model"].classes_)

### J.4 Model Technical Performance

In [ ]:
# Three reference points, because a model must beat something to be useful.
#
# Majority class: always predict the most common class in training.
# Persistence:    predict that the hazard in seven days equals today's.
# Seasonal:       predict the most common class for that day of the year.
majority_class = int(y_train.mode().iloc[0])

seasonal_lookup = (
    train.assign(day_of_year=train["date"].dt.dayofyear)
    .groupby("day_of_year")["whc_h7"]
    .agg(lambda s: s.mode().iloc[0])
)

predictions = {
    "Logistic regression": model.predict(X_test),
    "Majority class": np.full(len(y_test), majority_class),
    "Persistence": test["whc"].to_numpy(),
    "Seasonal mode": test["date"].dt.dayofyear.map(seasonal_lookup).fillna(majority_class).astype(int).to_numpy(),
}

comparison = pd.DataFrame([
    {"model": name, **modeling.evaluate_classification(y_test, pred, labels=CLASS_LABELS)}
    for name, pred in predictions.items()
]).set_index("model")

comparison[["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]].round(3)

In [ ]:
# Per-class recall: the operationally meaningful view.
recall_table = comparison[[f"recall_class_{c}" for c in CLASS_LABELS]]
recall_table.columns = CLASS_NAMES
recall_table.round(3)

In [ ]:
y_pred = predictions["Logistic regression"]

visualization.plot_multiclass_confusion_matrix(
    y_test, y_pred, labels=CLASS_LABELS, display_labels=CLASS_NAMES,
    title="Weather Hazard Category: confusion matrix (test set)",
)
plt.show()

# Normalised by true class, so each row reads as per-class recall.
visualization.plot_multiclass_confusion_matrix(
    y_test, y_pred, labels=CLASS_LABELS, display_labels=CLASS_NAMES,
    normalize="true",
    title="Weather Hazard Category: recall by true class (test set)",
)
plt.show()

In [ ]:
# How severe are the mistakes? Because the classes are ordinal, confusing
# Low with Extreme is far worse than confusing Low with Moderate, and a
# standard classification metric does not distinguish the two.
severity = pd.Series(np.abs(np.asarray(y_test) - np.asarray(y_pred)))
print("Distance between predicted and actual class:")
for distance, count in severity.value_counts().sort_index().items():
    print(f"  off by {distance}: {count:>5} days ({count / len(severity):.1%})")

print(f"\nExactly correct: {(severity == 0).mean():.1%}")
print(f"Within one class: {(severity <= 1).mean():.1%}")

In [ ]:
# Did the model catch the genuinely hazardous days?
hazardous = np.asarray(y_test) >= 2
flagged = np.asarray(y_pred) >= 2

print(f"Genuinely hazardous test days (High or Extreme): {hazardous.sum()}")
print(f"  of which flagged as hazardous: {(hazardous & flagged).sum()}"
      f"  ({(hazardous & flagged).sum() / max(hazardous.sum(), 1):.1%} recall)")
print(f"Days flagged as hazardous: {flagged.sum()}")
print(f"  of which genuinely were:  {(hazardous & flagged).sum()}"
      f"  ({(hazardous & flagged).sum() / max(flagged.sum(), 1):.1%} precision)")
print(f"\nFalse alarms: {(flagged & ~hazardous).sum()} days")
print(f"Missed hazards: {(hazardous & ~flagged).sum()} days")

### J.4.1 Is the failure a threshold problem or a signal problem?

A classifier that never predicts a rare class may still rank days correctly,
in which case lowering the decision threshold recovers it. This checks
whether that is the case here.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

# Probability the model assigns to a day being High or Extreme Risk.
probabilities = model.predict_proba(X_test)
hazard_probability = probabilities[:, 2] + probabilities[:, 3]
is_hazardous = np.asarray(y_test) >= 2

print(f"Test days {len(y_test):,}, hazardous {is_hazardous.sum()} ({is_hazardous.mean():.1%})")
print(f"Highest hazard probability assigned to any day: {hazard_probability.max():.3f}")
print()
print("threshold  alerts  caught  recall  precision  alerts/yr")
for threshold in [0.5, 0.3, 0.2, 0.15, 0.10, 0.075, 0.05]:
    flagged_at = hazard_probability >= threshold
    if flagged_at.sum() == 0:
        print(f"   {threshold:>5.3f}       0       0    0.0%          -          0")
        continue
    recall = (is_hazardous & flagged_at).sum() / is_hazardous.sum()
    precision = (is_hazardous & flagged_at).sum() / flagged_at.sum()
    print(f"   {threshold:>5.3f}  {flagged_at.sum():>6}  {(is_hazardous & flagged_at).sum():>6}"
          f"  {recall:>5.1%}  {precision:>9.1%}  {flagged_at.sum() / (len(y_test) / 365.25):>9.0f}")

In [ ]:
# The decisive test. These measure whether the model RANKS hazardous days
# above safe ones, independently of any threshold. A value of 0.5 for ROC
# AUC is what random guessing achieves.
print(f"ROC AUC (hazardous vs not): {roc_auc_score(is_hazardous, hazard_probability):.3f}")
print(f"Average precision:          {average_precision_score(is_hazardous, hazard_probability):.3f}"
      f"   (base rate {is_hazardous.mean():.3f})")

In [ ]:
# Has the class distribution itself shifted between training and test? The
# 26-year window was chosen to gather rare examples; this checks what that
# choice cost in exchange.
shares = pd.DataFrame({
    "train": train["whc_h7"].value_counts(normalize=True).reindex(CLASS_LABELS, fill_value=0),
    "test": test["whc_h7"].value_counts(normalize=True).reindex(CLASS_LABELS, fill_value=0),
})
shares.index = CLASS_NAMES
shares.round(3)

In [ ]:
# What does the model rely on? Multinomial logistic regression fits one set
# of coefficients per class, so permutation importance -- which measures the
# model as a whole on held-out data -- is the clearer view here.
permutation = importance.permutation_feature_importance(
    model, X_test, y_test, scoring="f1_macro",
    n_repeats=5, random_state=RANDOM_STATE,
)
permutation.head(15).round(4)

In [ ]:
visualization.plot_feature_importance(
    permutation, value_column="importance_mean", top_n=15,
    title="Weather Hazard Category: permutation importance (test set)",
)
plt.show()

In [ ]:
model_performance_explanations = """
The model does not work, and the diagnostics identify precisely why.

Selection on validation macro F1 chose the UNWEIGHTED model (C=0.1), which
achieves a test macro F1 of 0.241 and predicts High or Extreme Risk on zero
of the 1,454 test days. It misses all 73 genuinely hazardous days and raises
no false alarms, because it raises no alarms at all. As a hazard service it
has no value whatsoever.

That the tuning selected this model is itself a finding worth reporting. The
class-weighted alternative does flag hazardous days -- at C=0.001 it flags
659 days and catches 25 of the 73, a recall of 34% -- but its precision is
3.8%, and its macro F1 of 0.198 is LOWER than the model that never warns.
Balanced accuracy agrees, at 0.236 against 0.262. Both metrics chosen
specifically to resist class imbalance nonetheless prefer a model that
ignores the rare classes entirely, because the weighted model destroys
performance on the two common classes faster than it gains on the rare ones.
Section I argued that accuracy would hide this failure; it turns out macro
F1 and balanced accuracy hide it too.

The threshold sweep rules out the obvious remedy. If the model ranked days
correctly but set its decision boundary too conservatively, lowering the
threshold would recover the hazardous class. It does not. The highest hazard
probability assigned to any test day is 0.263, and thresholds down to 0.075
catch 2 of 73 hazards while raising 107 alerts.

The decisive measurement is the ranking quality. ROC AUC for hazardous
against non-hazardous days is 0.378, and average precision is 0.039 against
a base rate of 0.050. Both are BELOW what random guessing achieves. The
model has not learned a weak signal that a better threshold could exploit;
its ordering of days by hazard is, on the test period, actively
anti-correlated with reality. No threshold, no reweighting and no
recalibration can rescue a model whose ranking is worse than random.

Two causes are visible, and they compound.

The first was predicted in section D.1: weather variables retain almost no
correlation with conditions seven days ahead. The persistence baseline
confirms this -- knowing today's hazard category tells you very little about
the category a week later.

The second was not anticipated and matters more. The class distribution has
shifted materially between the training and test periods. Low Risk accounts
for 60% of training targets but only 48% of test targets, while Moderate
Risk rises from 39% to 52%. The 26-year window was chosen deliberately to
accumulate rare-class examples, accepting distribution drift as the price.
The comparison above suggests the price was too high: the model learned
relationships from a period that no longer describes the present, which is
consistent with the four-point decline in mean comfort observed over the
same span. A sub-random AUC is what you would expect from a model
confidently applying an outdated relationship.

Extreme Risk recall is zero in every configuration tested, as anticipated.
With four training examples this is not a surprise and should not be
reported as a model deficiency; it is a property of the data.

The severity breakdown offers the only encouraging note: 48.9% of days are
classified exactly and 96.5% within one class. That figure is not evidence
of skill, though. It reflects that 97% of days are Low or Moderate and the
model assigns nearly all days to those two classes, so it is almost always
within one class of the truth by construction.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='model_performance_explanations', value=model_performance_explanations)

### J.5 Business Impact from Current Model Performance

In [ ]:
# Translating the confusion matrix into operational terms: for every real
# hazard caught, how many false alarms does an operator absorb?
caught = int((hazardous & flagged).sum())
false_alarms = int((flagged & ~hazardous).sum())
missed = int((hazardous & ~flagged).sum())

print(f"Over {len(y_test):,} test days ({test['date'].min().date()} to {test['date'].max().date()}):")
print(f"  hazards caught:  {caught}")
print(f"  hazards missed:  {missed}")
print(f"  false alarms:    {false_alarms}")
if caught:
    print(f"\n  false alarms per hazard caught: {false_alarms / caught:.1f}")
print(f"  alerts raised per year: {flagged.sum() / (len(y_test) / 365.25):.0f}")

In [ ]:
business_impacts_explanations = """
The service, as built by this experiment, must not be deployed as an
operational hazard alert. It would raise no alerts at all while appearing to
function, which is the most dangerous possible failure mode for a safety
adjacent tool: silence indistinguishable from an all-clear.

Over the four-year test period the model missed all 73 genuinely hazardous
days and raised zero alerts. An operator integrating this endpoint and
receiving Low or Moderate Risk every day would reasonably infer that no
hazardous weather was expected, when in fact a High Risk day occurred
roughly every three weeks.

The class-weighted alternative is no more deployable, in the opposite
direction. Flagging 659 of 1,454 days -- 165 alerts a year, one day in
every two -- while being correct 3.8% of the time would exhaust operator
trust within weeks. Alert fatigue is a well-documented failure mode in
safety systems, and a service crying wolf every other day is worse than no
service, because it trains people to ignore warnings that might later be
correct.

Section A set out the cost asymmetry: a missed hazard is far worse than a
false alarm. That argument justifies accepting poor precision to gain
recall, and it is the basis on which the weighted model might have been
preferred despite scoring worse on every standard metric. The threshold and
AUC diagnostics remove that option. Trading precision for recall is only
rational when the model ranks days meaningfully, and at an AUC of 0.378 it
does not. There is no operating point on this curve worth choosing.

The honest conclusion is that a seven-day hazard forecast for Sydney is not
achievable with a linear model over local weather history, and the API must
say so. Three specific obligations follow:

The /model-metadata endpoint must report per-class recall, so that a
consumer can see the hazardous classes are not detected rather than having
to discover it. Reporting only an aggregate figure would be misleading.

The endpoint documentation must state that the hazard model is experimental
and must not be used as a safety input.

The model artefact is saved and the endpoint built so that the deployment
architecture can be validated end to end, since the service contract does
not depend on model quality. Replacing the artefact after experiment two
requires no change to the API.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='business_impacts_explanations', value=business_impacts_explanations)

### J.6 Save Model Artefact

In [ ]:
WHC_MODELS_DIR.mkdir(parents=True, exist_ok=True)

test_metrics = modeling.evaluate_classification(y_test, y_pred, labels=CLASS_LABELS)

artefact = {
    "pipeline": model,
    "recipe": WHC_RECIPE,
    "feature_columns": feature_columns,
    "history_days": features.required_history_days(WHC_RECIPE),
    "target": "Weather Hazard Category",
    "horizon_days": WHC_HORIZON,
    "algorithm": "Multinomial Logistic Regression",
    "hyperparameters": {"C": BEST_C, "class_weight": BEST_WEIGHT},
    "class_labels": CLASS_LABELS,
    "class_names": CLASS_NAMES,
    "training_window": [WHC_TRAIN_START, VALIDATION_START],
    "metrics": test_metrics,
    "feature_importance": (
        permutation.head(10).set_index("feature")["importance_mean"].round(5).to_dict()
    ),
}

path = WHC_MODELS_DIR / "whc_logistic_h7.joblib"
joblib.dump(artefact, path)
print(f"saved {path.name}  (macro F1 {test_metrics['macro_f1']:.3f})")

## H. Project Outcomes

In [ ]:
# Either 'Hypothesis Confirmed', 'Hypothesis Partially Confirmed' or
# 'Hypothesis Rejected'.
#
# Rejected: the model's ranking of hazardous days is worse than random
# (ROC AUC 0.378), so the failure is an absence of signal rather than a
# threshold or weighting problem.
experiment_outcome = "Hypothesis Rejected"

In [ ]:
# Do not modify this code
print_tile(size="h2", key='experiment_outcomes_explanations', value=experiment_outcome)

In [ ]:
experiment_results_explanations = """
The hypothesis was that a linear classifier with balanced class weights,
given lagged weather, rolling regime summaries and cyclical calendar
features, could usefully forecast the Weather Hazard Category seven days
ahead. It is rejected.

The evidence is unambiguous. The selected model predicts no hazardous days
at all, missing all 73 in the test period. The class-weighted alternative
flags 45% of all days to catch a third of the hazards, at 3.8% precision.
And the ranking diagnostic shows the failure is not one of calibration: with
a ROC AUC of 0.378 and average precision below the base rate, the model
orders days worse than chance would. There is no threshold at which this
model becomes useful.

Four insights follow.

The seven-day horizon is the binding constraint, exactly as section D.1
warned. Correlations between today's weather and conditions a week later are
near zero, and the persistence baseline performs accordingly. This is a
property of atmospheric predictability, not of the algorithm, and no
refinement of a model over these features will overcome it.

Distribution drift appears to have made matters actively worse. Low Risk
falls from 60% of training targets to 48% of test targets while Moderate
rises from 39% to 52%. The decision to use the full 26-year window, taken to
accumulate rare-class examples, traded relevance for volume and on this
evidence traded badly. A sub-random AUC is the signature of a model applying
a relationship that no longer holds. Testing a recent-only window is
therefore the highest-priority next step, even though it reduces High Risk
examples from 274 to roughly 158 and Extreme from 11 to 9.

Macro F1 and balanced accuracy both failed as selection criteria. Both were
chosen specifically to resist class imbalance, and both preferred a model
that never warns over one catching a third of hazards, because the weighted
model loses ground on the common classes faster than it gains on the rare
ones. The lesson is that no aggregate single number encoded the actual
business objective; the per-class recall table and the hazard summary did.
A future experiment should select on an explicitly operational criterion --
recall on hazardous classes subject to a maximum tolerable alert rate --
rather than on a general-purpose metric.

Extreme Risk is not learnable from this data. Four training examples cannot
support a class, and every configuration returned zero recall. This should
be treated as a documented limitation of the service rather than a target
for further modelling effort.

Next experiments, ranked by expected uplift:

1. A recent-only training window (2015 onwards), to test directly whether
   drift or horizon is the dominant cause. This is cheap, and the AUC below
   0.5 makes it the most informative single change available.
2. A gradient boosted tree ensemble. Hazardous days are defined by a
   conjunction -- heavy rain AND severe gusts AND total cloud together --
   which a linear model cannot represent without explicit interaction terms
   and a tree represents natively.
3. Reframing as a binary problem: hazardous against not hazardous, with the
   four-class output still reported as the specification requires. Merging
   High and Extreme raises the positive class from 0.12% to roughly 3% and
   gives the optimiser something it can work with.
4. External predictors -- synoptic pressure patterns, oceanic oscillation
   indices. Local surface observations may simply not contain seven-day
   information, and if experiments 1 to 3 fail this is the remaining avenue.

For deployment: the artefact is saved so the API and Render deployment can
be built and validated end to end, since the service contract is independent
of model quality. It must be deployed with explicit documentation that the
hazard model is experimental and unsuitable as a safety input, and with
per-class recall exposed through /model-metadata so that consumers can see
the limitation directly. The artefact should be replaced once a subsequent
experiment produces a model with better than random ranking.
"""

In [ ]:
# Do not modify this code
print_tile(size="h2", key='experiment_results_explanations', value=experiment_results_explanations)